# Lab 1 — Del crudo a la matriz de modelación · Financiera Andes
## Modelador de Riesgo de Crédito · Academia Bayes — **Entrega 1** (20% del curso)

**Se publica el miércoles 26 de agosto · fecha límite: domingo 30 de agosto, 23:59 (hora de Chile).**
Entrega: link a este notebook en Colab, **ejecutado de arriba a abajo sin errores**
(`Entorno de ejecución → Reiniciar y ejecutar todo`).

Este es el **laboratorio de la semana 1**: un solo notebook que cubre las clases 1 y 2.
Con la **Financiera Andes** (dataset B) vas a repetir lo que hicimos en clase con Banco
Austral, de punta a punta:

| | Parte | Qué construyes | Puntos |
|---|---|---|---|
| **A** | Clase 1 · Diseño y supuestos | Población, target, exclusiones y las cuatro muestras | 10 |
| **B** | Clase 2 · La fábrica de variables | Matriz y calidad | 8 + 4 bonus |

**Base: 18 puntos**; los ítems 11 y 12 son bonus (+4). `nota = mín(10; 1 + 9 × puntaje / 18)`.

**Puedes empezar hoy mismo por la parte A**, que ya vimos en la clase 1. La **parte B**
necesita la clase del jueves 27 — si la abres antes, vas a encontrarte con conceptos que
todavía no vimos. No hay problema en mirarla, pero no te frustres si no la entiendes aún.

Donde veas `# 👉 TU CÓDIGO`, completa; donde veas **✍️ Tu justificación**, escribe en texto
(se evalúa el argumento, no solo el código).

> Los datos son 100% sintéticos. La Financiera Andes es más riesgosa que el Banco Austral
> de la demo: no esperes los mismos números. Y como cada estudiante trabaja una **muestra
> personal**, tus resultados tampoco son los de tu compañero.

---
## 0. Identificación y muestra personal

Cada estudiante trabaja con una **muestra personal** del dataset (≈60% de los
clientes), determinada por su correo. Así tus números son únicos — la pauta
corrige rangos y método, no valores copiables.

⚠️ Usa el **mismo correo con el que te inscribiste** y no lo cambies durante el curso.

In [ ]:
ID_ALUMNO = "tu.correo@ejemplo.cl"   # ← EDITA con tu correo de inscripción

import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (9, 4.2)
plt.rcParams["axes.grid"] = True; plt.rcParams["grid.alpha"] = 0.3

assert ID_ALUMNO != "tu.correo@ejemplo.cl", "⚠️ Edita ID_ALUMNO con tu correo real"
SEMILLA_PERSONAL = int(hashlib.sha256(f"ANDES-C1-2026-B|{ID_ALUMNO.strip().lower()}".encode()).hexdigest()[:8], 16) % 10**6
print(f"Tu semilla personal: {SEMILLA_PERSONAL}")

In [ ]:
# Versión fija de los datos: evita que una actualización futura cambie los resultados del lab.
BASE = "https://raw.githubusercontent.com/nexolabs-gh/datos-riesgo-credito/feaa1968d0808c44ff228506523b706b677e3020"
PREFIJO = "andes"

clientes       = pd.read_parquet(f"{BASE}/{PREFIJO}_clientes.parquet")
solicitudes    = pd.read_parquet(f"{BASE}/{PREFIJO}_solicitudes.parquet")
comportamiento = pd.read_parquet(f"{BASE}/{PREFIJO}_comportamiento.parquet")
bureau         = pd.read_parquet(f"{BASE}/{PREFIJO}_bureau.parquet")

# ---- Muestra personal (NO modificar esta celda) ----
rng_p = np.random.default_rng(SEMILLA_PERSONAL)
ids = np.sort(clientes["id_cliente"].unique())
mis_clientes = set(rng_p.choice(ids, size=int(len(ids) * 0.60), replace=False))
clientes       = clientes[clientes["id_cliente"].isin(mis_clientes)].reset_index(drop=True)
solicitudes    = solicitudes[solicitudes["id_cliente"].isin(mis_clientes)].reset_index(drop=True)
comportamiento = comportamiento[comportamiento["id_cliente"].isin(mis_clientes)].reset_index(drop=True)
bureau         = bureau[bureau["id_cliente"].isin(mis_clientes)].reset_index(drop=True)
print(f"Tu muestra: {len(clientes):,} clientes · {len(solicitudes):,} solicitudes")

### Primero el diccionario de datos, siempre

Nadie modela columnas que no sabe qué significan. En un banco real el diccionario
de datos se **exige antes** de escribir la primera línea, y si no existe, se
construye entrevistando a quien administra el sistema de origen. Dice qué mide
cada campo, en qué unidad, con qué frecuencia se actualiza y —lo decisivo para
riesgo— **en qué momento del tiempo se llenó**. Adivinar por el nombre de la
columna es la forma más barata de meter una fuga de información en un modelo.

El diccionario de Financiera Andes está publicado junto a los datos:
[`diccionario_andes.md`](https://github.com/nexolabs-gh/datos-riesgo-credito/blob/main/diccionario_andes.md)

La celda siguiente lo muestra **completo — las cuatro tablas que acabamos de
cargar** — y deja definida `mostrar_diccionario(...)` para volver a consultar una
tabla puntual más adelante (por ejemplo `mostrar_diccionario("solicitudes")`).

> ⚠️ Léelo completo antes de decidir qué columnas entran al modelo. Algunas de las columnas problemáticas de este dataset **se delatan en su propia descripción** — la más peligrosa, no: esa hay que razonarla contra el punto de observación.

In [ ]:
import urllib.request

from IPython.display import Markdown, display

URL_DICCIONARIO = f"{BASE}/diccionario_{PREFIJO}.md"
_DICCIONARIO = urllib.request.urlopen(URL_DICCIONARIO).read().decode("utf-8")


def mostrar_diccionario(tabla=None):
    """Muestra el diccionario de datos oficial; sin argumento, lo muestra completo."""
    if tabla is None:
        display(Markdown(_DICCIONARIO))
        return
    secciones = _DICCIONARIO.split("\n## ")[1:]
    elegidas = [s for s in secciones if s.lower().startswith(tabla.lower())]
    if not elegidas:
        disponibles = [s.split("(")[0].strip() for s in secciones]
        raise ValueError(f"No existe la tabla {tabla!r}. Disponibles: {disponibles}")
    display(Markdown("## " + elegidas[0]))


mostrar_diccionario()

---
# Parte A · Diseño y supuestos
### Clase 1 · 10 puntos

Cierras esta parte con la **población de modelación** de la Financiera Andes: quién entra,
qué es un default, quién queda fuera y cómo se reparten las muestras. Es el contrato del
proyecto — todo lo que viene después se apoya acá.

---
## 1. Reconocimiento del terreno (1 punto)

Antes de modelar, entiende qué te entregaron. Responde **con código y con texto**.

In [ ]:
# 👉 TU CÓDIGO: para cada tabla muestra filas × columnas, y para 'solicitudes'
# el rango de fechas y la tasa de aprobación. Para 'clientes', el % de missing
# de renta_liquida.


**✍️ Tu justificación (1a):** ¿cuál es la unidad de análisis del modelo que
construiremos y qué población representa? ¿Modelaremos toda la demanda o solo
una parte? ¿Qué implicancia tiene eso?



---
## 2. Sustenta la definición de default (2 puntos)

Construye la **matriz de roll rates** mensual de la Financiera Andes (bandas:
al día / 1–29 / 30–59 / 60–89 / 90+) y a partir de ella **defiende o cuestiona**
usar 90+ como definición de default en esta cartera.

In [ ]:
def banda_mora(dias):
    # 👉 TU CÓDIGO: np.select con las 5 bandas estándar
    ...

# 👉 TU CÓDIGO: matriz de transición banda (mes t) -> banda (mes t+1), en %
# Pista: ordena por cliente y mes, usa groupby(...)["banda"].shift(-1) y pd.crosstab
roll = ...
roll

**✍️ Tu justificación (2a):** ¿dónde está el "punto de no retorno" en esta
cartera? Compara la probabilidad de cura entre bandas y concluye si 90+ es una
definición razonable de default para la Financiera Andes.



---
## 3. Construye el target (3 puntos — el corazón de esta entrega)

Recuerda la **máquina del tiempo** de la clase: para etiquetar no necesitas
adivinar el futuro — te paras en el pasado (la fecha de cada solicitud) y su
"futuro" de 12 meses **ya ocurrió y está en el panel**. Y cada solicitud lleva
**su propia ventana**: la de enero se observa hasta el enero siguiente; la de
junio, hasta el junio siguiente.

Implementa `construir_target` con la definición del curso:
**malo = alcanzar 90+ DPD dentro de los 12 meses posteriores al punto de
observación** (mes de la solicitud). Marca además los **indeterminados**
(peor mora 30–89) y deja con target nulo a quien no tenga ventana completa.

Regla de oro: la ventana va de $t_0+1$ a $t_0+12$. **Ni un mes más, ni el mes
de la solicitud.**

In [ ]:
# Preparación (se entrega hecha): mora en formato ancho cliente × mes
mora_w = comportamiento.pivot_table(index="id_cliente", columns="mes",
                                    values="dias_mora", aggfunc="first")
meses = list(mora_w.columns)
pos_mes = {m: k for k, m in enumerate(meses)}
M = mora_w.to_numpy()
fila_de = {c: k for k, c in enumerate(mora_w.index)}

cursadas = solicitudes[solicitudes["aprobada"]].copy()
cursadas["_t"] = cursadas["fecha_solicitud"].map(pos_mes)
cursadas["_i"] = cursadas["id_cliente"].map(fila_de)

In [ ]:
def construir_target(cursadas, M, n_meses_panel, ventana=12, umbral_malo=90, umbral_gris=30):
    out = cursadas.copy()
    peor = np.full(len(out), np.nan)
    for k, (i, t0) in enumerate(zip(out["_i"], out["_t"])):
        # 👉 TU CÓDIGO:
        #   1) calcula el índice final de la ventana (t0 + ventana)
        #   2) si la ventana cabe completa en el panel, guarda en peor[k]
        #      el máximo de M[i, ...] SOLO dentro de la ventana de desempeño
        ...
    out["peor_dpd_12m"] = peor
    # 👉 TU CÓDIGO: columnas 'malo' (1/0, NaN si sin ventana) e 'indeterminado'
    ...
    return out

etiquetadas = construir_target(cursadas, M, len(meses))
etiquetadas[["fecha_solicitud", "peor_dpd_12m", "malo", "indeterminado"]].head()

In [ ]:
# ---- Verificación automática (NO modificar): sanidad del target ----
_completa = etiquetadas[etiquetadas["malo"].notna()]
assert etiquetadas.loc[etiquetadas["fecha_solicitud"] > "2025-06", "malo"].isna().all(), \
    "❌ Cosechas posteriores a 2025-06 no tienen ventana completa: su target debe ser NaN"
assert 0.05 <= _completa["malo"].mean() <= 0.20, \
    f"❌ Tasa de malos fuera de rango razonable: {_completa['malo'].mean():.1%} — revisa la ventana"
assert (_completa.loc[_completa["indeterminado"], "malo"] == 0).all(), \
    "❌ Un indeterminado no puede ser malo a la vez"
print(f"✅ Target OK · tasa de malos (ventana completa): {_completa['malo'].mean():.1%}")

---
## 4. Población de modelación: cliente único, exclusiones y columnas prohibidas (2 puntos)

**(4a) Cliente único.** Un cliente activo puede tener varias solicitudes en el
período de modelación (2024-09 → 2025-06). **Regla del curso**: cada cliente
entra **una sola vez**, con su **primera** solicitud del período; si esa fila
después cae excluida, el cliente sale (no se reemplaza por otra solicitud suya).
Cuenta cuántas filas repetidas descartas.

**(4b) Exclusiones.** Sobre esa población, aplica y reporta la cascada de
exclusiones: ventana completa, sin fraude, sin indeterminados, y clientes con
≥ 6 meses de historia previa. Justifica cada una.

**(4c) Columnas prohibidas.** En estas tablas hay **al menos dos columnas** que
un modelo de admisión jamás podría usar como predictor porque contienen
información posterior al punto de observación. **Identifícalas, demuestra el
problema con datos** (por ejemplo, mide su poder predictivo) y explica el
mecanismo de la fuga. Pista de la sutil: mira cómo evoluciona el **cupo** de los
clientes que se deterioran.

In [ ]:
# 👉 TU CÓDIGO (4a): filtra el período de modelación (2024-09 → 2025-06) y
# deja UNA fila por cliente (su primera solicitud del período). Reporta cuántas
# filas repetidas descartas y qué % eran.
periodo = ...


**✍️ Tu justificación (4a):** ¿por qué no dejamos que un cliente aparezca dos
veces? Menciona qué pasaría con un split aleatorio DEV/HO por fila, y qué
alternativas existen cuando la cartera es demasiado chica para este lujo.



In [ ]:
# 👉 TU CÓDIGO (4b): aplica la cascada de exclusiones sobre `periodo` y reporta
# cuántos créditos elimina cada paso. Población final: ventana completa, sin
# fraude, sin indeterminados, y clientes con ≥ 6 meses de historia previa.
entrenables = ...


In [ ]:
# 👉 TU CÓDIGO (4c): demuestra la fuga de la columna "obvia" de solicitudes
# (mide, p. ej., su AUC contra el target) y evidencia la sutil con un gráfico
# o tabla (cupo promedio alrededor del inicio de la mora).


**✍️ Tu justificación (4b):** explica por qué aplicaste cada exclusión.

**✍️ Tu justificación (4c):** nombra las columnas prohibidas, el mecanismo de cada fuga y cómo la detectarías en un proyecto real.



---
## 5. Cosechas y esquema muestral (2 puntos)

**(5a)** Tabla y gráfico de **tasa de malos por cosecha** (2024-09 → 2025-06).
¿Qué observas hacia las cosechas finales? ¿Qué implicancia tiene para el modelo?

**(5b)** Construye las muestras del curso y reporta n, malos y tasa por muestra:
- **DEV/HO**: cosechas 2024-09 → 2025-01, split aleatorio 75/25 (usa `SEMILLA_PERSONAL`)
- **OOT**: cosechas 2025-02 → 2025-06
- **TTD**: solicitudes cursadas desde 2025-07 con **≥ 6 meses de historia** (sin target).
  El mínimo de historia es el mismo de la población entrenable: sin él no hay ventanas
  hacia atrás que calcular en la parte B.

In [ ]:
# 👉 TU CÓDIGO (5a): tasa de malos por cosecha, tabla + gráfico de barras


In [ ]:
# 👉 TU CÓDIGO (5b): columna 'muestra' (DEV/HO/OOT) + definición de TTD + tabla resumen
rng = np.random.default_rng(SEMILLA_PERSONAL)
...

In [ ]:
# ---- Verificación automática (NO modificar): esquema muestral ----
assert entrenables["id_cliente"].is_unique, \
    "❌ Hay clientes repetidos en la población de modelación: aplica la regla de cliente único (4a)"
assert set(entrenables["muestra"].unique()) == {"DEV", "HO", "OOT"}
assert entrenables.loc[entrenables["muestra"] == "OOT", "fecha_solicitud"].min() >= "2025-02", \
    "❌ OOT contaminada con cosechas de desarrollo"
assert entrenables.loc[entrenables["muestra"].isin(["DEV", "HO"]), "fecha_solicitud"].max() <= "2025-01", \
    "❌ DEV/HO contiene cosechas del período OOT"
_p = entrenables["muestra"].value_counts(normalize=True)
assert 0.65 <= _p.get("DEV", 0) / (_p.get("DEV", 0) + _p.get("HO", 0)) <= 0.85, "❌ split 75/25 fuera de rango"
_alta_ttd = clientes.set_index("id_cliente")["fecha_alta_cliente"].map(
    lambda s: (pd.Period(s, freq="M") - pd.Period(meses[0], freq="M")).n)
_hist_ttd = ttd["_t"] - ttd["id_cliente"].map(_alta_ttd)
assert (ttd["fecha_solicitud"] >= "2025-07").all(), "❌ TTD contiene solicitudes anteriores a 2025-07"
assert (_hist_ttd >= 6).all(), "❌ TTD contiene clientes con menos de 6 meses de historia"
assert ttd["malo"].isna().all(), "❌ TTD no debe tener target observado"
print("✅ Esquema muestral OK")

**✍️ Tu lectura (5a):** describe el cambio de la tasa por cosecha y su
implicancia para el modelo.

**✍️ Tu justificación (5c):** explica con tus palabras el rol de cada muestra
(DEV, HO, OOT, TTD) y por qué la OOT "no se toca" hasta la validación.



---
# Puente · de la parte A a la parte B

La celda que sigue deja a **todo el curso en el mismo punto de partida** para la fábrica de
variables: reconstruye la población sobre tu muestra personal y la deja en el formato que
usa la parte B. **No la modifiques.**

Hace tres cosas, y conviene entender por qué:

1. **Reconstruye el terreno de la parte A** — target, cliente único, exclusiones y muestras.
   Es la solución de referencia: si tu parte A quedó a medias, **la parte B corre igual** y no
   pierdes la entrega. Tu parte A se corrige por tu propio código y tus justificaciones.
2. **Compara su resultado con el tuyo** y te dice si coinciden. Es tu primera corrección
   automática: si difieren mucho, vuelve a la sección 4 antes de seguir.
3. **Amplía la población**: la parte A se quedó con los créditos *entrenables* (sin
   indeterminados). La fábrica, en cambio, calcula variables para **todos** — indeterminados
   y TTD incluidos — porque en producción hay que puntuar a todo el que llega. Los
   indeterminados se excluyen recién en el **screening** (sección 11), que es donde se
   necesita un target definido.

In [ ]:
mora_w = comportamiento.pivot_table(index="id_cliente", columns="mes",
                                    values="dias_mora", aggfunc="first")
meses = list(mora_w.columns)
pos_mes = {m: k for k, m in enumerate(meses)}
M = mora_w.to_numpy()
fila_de = {c: k for k, c in enumerate(mora_w.index)}

cursadas = solicitudes[solicitudes["aprobada"]].copy()
cursadas["_t"] = cursadas["fecha_solicitud"].map(pos_mes)
cursadas["_i"] = cursadas["id_cliente"].map(fila_de)

def construir_target(cursadas, M, n_meses, ventana=12, umbral_malo=90, umbral_gris=30):
    out = cursadas.copy()
    peor = np.full(len(out), np.nan)
    for k, (i, t0) in enumerate(zip(out["_i"], out["_t"])):
        fin = t0 + ventana
        if fin <= n_meses - 1:
            peor[k] = M[i, t0 + 1: fin + 1].max()
    out["peor_dpd_12m"] = peor
    out["malo"] = np.where(np.isnan(peor), np.nan, (peor >= umbral_malo).astype(float))
    out["indeterminado"] = (peor >= umbral_gris) & (peor < umbral_malo)
    return out

etiquetadas = construir_target(cursadas, M, len(meses))

antig = clientes.set_index("id_cliente")["fecha_alta_cliente"].map(
    lambda s: (pd.Period(s, freq="M") - pd.Period(meses[0], freq="M")).n)

periodo = etiquetadas[etiquetadas["fecha_solicitud"].between("2024-09", "2025-06")]
periodo = periodo.sort_values("fecha_solicitud", kind="stable").drop_duplicates(
    "id_cliente", keep="first").copy()
periodo["antiguedad_meses"] = periodo["_t"] - periodo["id_cliente"].map(antig)

base = periodo[periodo["malo"].notna() & ~periodo["marca_fraude"]
               & (periodo["antiguedad_meses"] >= 6)].copy()
rng = np.random.default_rng(SEMILLA_PERSONAL)
base["muestra"] = np.where(base["fecha_solicitud"] <= "2025-01",
                           np.where(rng.random(len(base)) < 0.75, "DEV", "HO"), "OOT")

ttd = etiquetadas[etiquetadas["fecha_solicitud"] >= "2025-07"].copy()
ttd["antiguedad_meses"] = ttd["_t"] - ttd["id_cliente"].map(antig)
ttd = ttd[ttd["antiguedad_meses"] >= 6]
ttd["muestra"] = "TTD"

poblacion = pd.concat([base, ttd], ignore_index=True)
print(poblacion["muestra"].value_counts().reindex(["DEV", "HO", "OOT", "TTD"]).to_string())
print(f"\nTasa de malos (sin indeterminados): "
      f"{base.loc[~base['indeterminado'], 'malo'].mean():.1%}")

# ---- ¿Coincide con TU parte A? (esta es tu primera corrección automática) ----
try:
    _tuyo, _ref = len(entrenables), len(base[~base["indeterminado"]])
    _d = abs(_tuyo - _ref) / _ref
    _v = "✅ coinciden" if _d < 0.02 else f"⚠️ difieren {_d:.0%} — revisa tu sección 4 antes de seguir"
    print(f"\nTu parte A: {_tuyo:,} entrenables · referencia: {_ref:,}  → {_v}")
except (NameError, TypeError):
    print("\n⚠️ No encontré `entrenables` de la parte A. La parte B corre igual; "
          "revisa tu sección 4 cuando puedas.")

---
# Parte B · La fábrica de variables
### Clase 2 · 8 puntos + 4 bonus

Ahora conviertes esas cuatro tablas crudas en una **matriz de modelación**: una fila por
solicitud, decenas de columnas candidatas, todas ancladas al punto de observación. Y haces
el primer filtro con WoE e IV.

> ⚠️ Esta parte necesita la **clase del jueves 27**.

---
## 6. El panel en formato ancho (1 punto)

Para calcular ventanas hacia atrás necesitas, por cada serie del panel, una
**matriz cliente × mes** alineada con `mora_w`.

**(6a)** Completa `pivotar()` y construye el diccionario `PANEL` con las series
del comportamiento y del bureau.

**(6b)** Agrega las **series derivadas**: utilización de línea y de tarjeta
(calculadas **mes a mes**, no como cociente de promedios), deuda interna y deuda
total.

In [ ]:
def pivotar(df, columna):
    # 👉 TU CÓDIGO: pivotea `columna` a matriz cliente × mes y devuélvela como
    # array de numpy, REINDEXADA a mora_w.index y a `meses` (para que la fila i
    # y la columna t signifiquen lo mismo en todas las matrices).
    ...

SERIES_COMPORTAMIENTO = ["dias_mora", "saldo_linea", "cupo_linea", "saldo_tc", "cupo_tc",
                         "saldo_consumo", "monto_pagado", "monto_facturado",
                         "abonos_cuenta", "saldo_ahorro", "n_productos"]
SERIES_BUREAU = ["deuda_otras_inst", "n_otras_inst", "consultas_mes", "peor_mora_sistema"]

# 👉 TU CÓDIGO: arma el diccionario PANEL con todas las series
PANEL = ...


In [ ]:
# 👉 TU CÓDIGO (6b): series derivadas calculadas MES A MES
# - uso_linea = saldo_linea / cupo_linea   (NaN si el cupo es 0)
# - uso_tc    = saldo_tc / cupo_tc         (NaN si el cupo es 0)
# - deuda_interna = saldo_linea + saldo_tc + saldo_consumo
# - deuda_total   = deuda_interna + deuda_otras_inst
# - en_mora / en_mora_sistema: indicadores 0/1 por mes (útiles para contar y para recencia)


**✍️ Tu justificación (6c):** ¿por qué la utilización se calcula **mes a mes**
y no como `promedio(saldo) / promedio(cupo)`? Da un ejemplo numérico donde ambas
formas den resultados distintos.



---
## 7. El motor de ventanas (2 puntos — el corazón del lab)

Implementa `agregar_ventana`. Para una solicitud con punto de observación $t_0$
y ventana de $k$ meses, la ventana es:

$$[\,t_0-k,\; t_0-1\,]$$

⚠️ **La trampa está en el índice.** En numpy, `matriz[i, a:b]` **no incluye** la
columna `b`. Si escribes `t0+1` como límite superior, estás metiendo el mes de la
solicitud dentro de la ventana — y contaminas **todas** tus variables.

Implementa también los cinco agregadores. Ojo con dos:
- `delta`: variación relativa entre los **extremos** de la ventana, `(fin/inicio) − 1`.
  Protege la división con `np.maximum(inicio, 1)`.
- `recencia`: meses desde el último mes con evento. Si **no hubo** evento en la
  ventana, devuelve `k+1` (valor censurado) — nunca el mismo valor que
  "el evento fue al principio de la ventana".

In [ ]:
def agregar_ventana(matriz, filas, t0s, k, funcion):
    """Agrega `matriz` sobre la ventana [t0-k, t0-1] de cada solicitud."""
    salida = np.full(len(filas), np.nan)
    for t0 in np.unique(t0s):
        sel = t0s == t0
        # 👉 TU CÓDIGO:
        #   1) calcula el inicio de la ventana (cuidado con el borde: no puede ser < 0)
        #   2) aplica `funcion` al slice correcto de `matriz` y guárdalo en salida[sel]
        ...
    return salida


def prom(V):  ...   # 👉 promedio ignorando NaN
def vmax(V):  ...   # 👉 máximo ignorando NaN
def suma(V):  ...   # 👉 suma ignorando NaN

def delta(V):
    # 👉 TU CÓDIGO: (último / primero) - 1, protegiendo el denominador
    ...

def recencia(V):
    # 👉 TU CÓDIGO: meses desde el último mes con evento (>0);
    # si no hubo evento en la ventana, devuelve k+1
    ...

AGREGADORES = {"prom": prom, "max": vmax, "suma": suma, "delta": delta, "recencia": recencia}

In [ ]:
# ---- Verificación automática (NO modificar): el ancla temporal ----
filas_p = poblacion["_i"].to_numpy()
t0s_p   = poblacion["_t"].to_numpy()

# Matriz de prueba donde el valor de cada celda ES el índice del mes.
# Así, sobre la ventana [t0-k, t0-1]:  máximo = t0-1  y  mínimo = t0-k.
_M_test = np.tile(np.arange(len(meses), dtype=float), (PANEL["dias_mora"].shape[0], 1))

_maximos = agregar_ventana(_M_test, filas_p, t0s_p, 3, vmax)
assert np.allclose(_maximos, t0s_p - 1), (
    "❌ FUGA: el último mes de tu ventana es "
    f"t0+{int(np.median(_maximos - t0s_p))}, y debe ser t0-1. "
    "Revisa el límite superior del slice: matriz[filas, inicio:t0] NO incluye t0")

_minimos = agregar_ventana(_M_test, filas_p, t0s_p, 3, lambda V: np.nanmin(V, axis=1))
assert np.allclose(_minimos, np.maximum(t0s_p - 3, 0)), \
    "❌ El primer mes de la ventana de 3 meses debe ser t0-3"

_r = recencia(np.array([[0., 0., 0.], [1., 0., 0.], [0., 0., 1.]]))
assert list(_r) == [4, 3, 1], f"❌ recencia mal implementada: {list(_r)} (esperado [4, 3, 1])"
print("✅ Ancla temporal correcta: ventana = [t0-k, t0-1] · recencia bien censurada")

---
## 8. Declara tu fábrica (2 puntos)

Escribe la lista `FABRICA` con tuplas `(serie, agregador, ventanas, plantilla)`.

**Requisitos mínimos de la entrega:**
- Al menos **40 variables de ventana**.
- Al menos **4 familias** distintas (mora propia, utilización, deuda, pagos,
  ahorro/ingreso, bureau, consultas).
- Las tres ventanas **3, 6 y 12 meses** presentes.
- Al menos **3 agregadores** distintos.
- Convención de nombres `<concepto>_<agregador>_<ventana>` respetada.

Añade después las **fotos** al último cierre disponible ($t_0-1$).

In [ ]:
# 👉 TU CÓDIGO: declara tu fábrica (mínimo 40 variables, ≥4 familias, ≥3 agregadores)
FABRICA = [
    ("dias_mora", "max", (3, 6, 12), "dias_mora_max_{k}m"),
    # ... completa
]

META = ["id_solicitud", "id_cliente", "fecha_solicitud", "_t", "_i",
        "muestra", "malo", "indeterminado", "peor_dpd_12m"]
X = poblacion[META + ["antiguedad_meses", "monto_solicitado", "plazo_meses",
                      "destino", "canal"]].copy()

# 👉 TU CÓDIGO: recorre FABRICA y crea cada variable con agregar_ventana


In [ ]:
# 👉 TU CÓDIGO: fotos al último cierre disponible (t0-1).
# Mínimo: dias_mora, saldo_linea, cupo_linea, saldo_ahorro, deuda_otras_inst,
# uso_linea, uso_tc. Nómbralas con el sufijo _ult
t_ult = ...


---
## 9. Ratios de negocio (1,5 puntos)

Construye **al menos 4 ratios** de negocio. Sugerencias (puedes proponer otros):
utilización ya la tienes; faltan carga financiera, pago sobre facturación,
deuda interna sobre deuda del sistema, ahorro sobre ingreso, cuota estimada
sobre ingreso.

**Decide y declara tu denominador de ingreso**: ¿renta declarada o abonos
observados? Justifícalo — es exactamente el tipo de supuesto que un validador
va a buscar en tu informe.

In [ ]:
# 👉 TU CÓDIGO: al menos 4 ratios de negocio.
# Protege SIEMPRE los denominadores con np.maximum(denominador, 1)


**✍️ Tu justificación (9a):** para cada ratio que construiste, una línea:
**qué hipótesis de riesgo captura**. Y explica qué denominador de ingreso
elegiste y por qué.



In [ ]:
# ---- Verificación automática (NO modificar): requisitos de la fábrica ----
_pred = [c for c in X.columns if c not in META]
_vent = [c for c in _pred if any(c.endswith(f"_{k}m") for k in (3, 6, 12))]
assert len(_vent) >= 40, f"❌ Solo {len(_vent)} variables de ventana: se exigen al menos 40"
for _k in (3, 6, 12):
    assert any(c.endswith(f"_{_k}m") for c in _pred), f"❌ Falta la ventana de {_k} meses"
_familias = sum(any(p in c for c in _pred) for p in
                ["mora", "uso_", "deuda", "pagos", "ahorro", "consultas", "abonos"])
assert _familias >= 4, f"❌ Solo {_familias} familias detectadas: se exigen al menos 4"
_ratios = [c for c in RATIOS if c in X.columns]
assert len(_ratios) >= 4, f"❌ Solo {len(_ratios)} ratios de negocio: se exigen al menos 4"
print(f"✅ Fábrica OK · {len(_vent)} variables de ventana · {_familias} familias · {len(_ratios)} ratios")

---
## 10. Diccionario y reporte de calidad (1,5 puntos)

**(10a)** Genera el **diccionario de variables** a partir de tu `FABRICA`
(variable, serie de origen, agregador, ventana). Como la fábrica es declarativa,
esto se genera solo — no lo escribas a mano.

**(10b)** Produce un **reporte de calidad**: por variable, `% missing`, mínimo,
mediana, p99, máximo y `% en el valor más frecuente` (la masa puntual).

**(10c)** Identifica en TU matriz al menos **un missing estructural** y **un
missing informativo (MNAR)**, y demuestra la diferencia con datos.

In [ ]:
# 👉 TU CÓDIGO (10a): diccionario generado desde FABRICA
diccionario = ...
diccionario.head(10)

In [ ]:
# 👉 TU CÓDIGO (10b): reporte de calidad de todas tus variables predictoras
calidad = ...
calidad[calidad["missing_%"] > 0].sort_values("missing_%", ascending=False)

In [ ]:
# 👉 TU CÓDIGO (10c): demuestra con datos un missing ESTRUCTURAL y uno INFORMATIVO


**✍️ Tu justificación (10d):** explica la diferencia entre los dos missing que
encontraste y qué harías con cada uno. ¿Por qué imputar la renta por la media
sería un error en este caso?



---
## 11. ⭐ BONUS · Screening univariado: binning, WoE e IV (3 puntos)

**(11a)** Implementa `binear()` y `tabla_woe()`. Requisitos:
- `MISSING` es **siempre** un bin propio.
- Una masa puntual dominante (por ejemplo, muchos ceros) va en **su propio bin**.
- El resto, a cuantiles (4–6 bins).
- Usa suavizado (+0,5) para evitar infinitos cuando un bin no tiene malos.

**(11b)** Calcula el IV de **todas** tus candidatas — **solo sobre DEV** — y
muestra el top 20 con su clasificación de industria.

⚠️ **Invalidante**: calcular el IV sobre la muestra completa (DEV+HO+OOT).

In [ ]:
def binear(x, bins=5, umbral_moda=0.35):
    # 👉 TU CÓDIGO: devuelve (etiquetas_de_bin, orden_de_los_bins)
    # - categóricas: cada nivel un bin
    # - masa puntual > umbral_moda: bin propio
    # - resto: cuantiles
    # - MISSING: siempre bin propio
    ...

def tabla_woe(x, y, bins=5):
    # 👉 TU CÓDIGO: tabla con n, malos, buenos, tasa_malos, woe, iv_aporte
    # y devuelve (tabla, iv)
    ...

# La muestra de ajuste: DEV, sin indeterminados
dev = ...
ho  = ...


In [ ]:
# 👉 TU CÓDIGO (11b): IV de todas las candidatas SOBRE DEV + top 20 + clasificación
ivs = ...

# 👉 TU CÓDIGO (11c): muestra la tabla WoE de una variable importante


In [ ]:
# ---- Verificación automática (NO modificar): screening ----
assert len(ivs) >= 40, "❌ Faltan variables en el screening"
_t, _iv = tabla_woe(dev["renta_liquida"], dev["malo"])
assert "MISSING" in _t.index, "❌ El bin MISSING debe existir: no imputes la renta"
assert abs(_t.loc["MISSING", "woe"]) > 0.01, "❌ El bin MISSING debe tener su propio WoE"
assert len(dev) < len(X[X["malo"].notna() & ~X["indeterminado"]]), \
    "❌ Estás calculando el IV sobre más que DEV"
_iv_ruido = tabla_woe(dev["sexo"], dev["malo"])[1]
assert _iv_ruido < 0.05, f"❌ 'sexo' debería tener IV ≈ 0, obtuviste {_iv_ruido:.3f}"
print("✅ Screening OK · el MISSING está tratado como categoría · IV calculado solo en DEV")

**✍️ Tu justificación (11c):** muestra la tabla WoE de **una** variable que te
parezca importante y explícala como si estuvieras frente al comité: qué dice cada
bin, si la relación es monótona y qué harías si no lo fuera.



---
## 12. ⭐ BONUS · Auditoría del top: ¿señal o fuga? (1 punto)

En la clase vimos que un IV muy alto es una **alerta**, no un logro.

**(12a)** Toma las **3 variables de IV más alto** de tu ranking y audítalas.
Para cada una responde con evidencia:
1. ¿Su ventana termina en $t_0-1$?
2. ¿Existiría, con esos valores, el día de la decisión?
3. ¿Tiene una explicación de negocio?
4. ¿Se mantiene su IV fuera de DEV (en hold-out)?

**(12b)** Demuestra con TU dataset el costo de un error de anclaje: toma una
variable y recalcúlala con la ventana **corrida dos meses hacia adelante**
(que termine en $t_0+1$ en vez de $t_0-1$). Compara los IV.

In [ ]:
# 👉 TU CÓDIGO (12a): tabla comparativa IV en DEV vs IV en HO para tu top 3


In [ ]:
# 👉 TU CÓDIGO (12b): recalcula una variable con la ventana corrida dos meses
# (que termine en t0+1 en vez de t0-1) y compara su IV con la versión bien anclada


**✍️ Tu justificación (12c):** ¿tus 3 variables del top son señal legítima o
fuga? Argumenta cada una. Y explica: si en un proyecto real te entregan un
modelo con Gini 0,95, ¿qué es lo primero que revisas?



---
## Cierre y entrega

**Checklist antes de entregar:**
- [ ] `Reiniciar y ejecutar todo` corre sin errores y **todos los ✅ aparecen**
- [ ] Tu `ID_ALUMNO` es tu correo real de inscripción
- [ ] **Parte A** — las 7 justificaciones escritas (1a, 2a, 4a, 4b, 4c, 5a, 5c) están completas
- [ ] **Parte B base** — las 3 justificaciones escritas (6c, 9a, 10d) están completas
- [ ] Tu matriz tiene al menos 40 variables de ventana, 4 familias y 4 ratios
- [ ] **Si haces el bonus:** 11c y 12c están respondidas y el IV está calculado **solo sobre DEV**

**Cómo se corrige:** base de 18 puntos + hasta 4 de bonus; `nota = mín(10; 1 + 9 × puntaje / 18)`. Las justificaciones
escritas pesan tanto como el código: un resultado correcto sin argumento no puntúa completo,
y un argumento sólido rescata buena parte de un cálculo que no salió.

> 💾 **Guarda este notebook.** La matriz `X` que construiste hoy es el insumo directo del
> **Lab 2** (selección multivariada, modelo y scorecard), que abre la Entrega 2 el martes 1
> de septiembre. Por eso esta entrega vence ahora y no después: el feedback tiene que
> llegarte **antes** de que construyas el modelo encima de esta matriz.